# Image Segmentation with DeeplabV3: 복강경 13클래스

균열 segmentation은 픽셀이 균열이냐 아니냐 두 클래스였다. 이번 장은 복강경 담낭절제술(laparoscopic cholecystectomy) 장면의 13클래스다. 간(liver)·담낭(gallbladder)·지방(fat)·도구가 한 화면에 섞이고 담낭관(cystic duct)·간정맥(hepatic vein)은 픽셀이 1%도 안 되는 경우가 많다.

**런타임 → 런타임 유형 변경 → GPU(T4)** 를 먼저 선택한다.

zip에는 학습·검증·테스트 슬라이스와 학습해 둔 DeepLabV3+ 가중치(`runs/best.pt`)가 있다. 7번 평가와 8번 시연은 이 파일을 읽는다. 학습·추론 코드는 이 노트북 셀에 있다.
`/content`는 세션이 끝나면 비므로 zip은 드라이브에 남겨 둔다.


## 1. 데이터 zip

드라이브에서 해당 zip을 `링크가 있는 모든 사용자`로 공유한 뒤, 링크를 `ZIP_URL`에 넣는다.

파일이 100MB보다 크면 `wget`이 구글 확인 페이지를 받는다. 드라이브 링크는 `gdown`, 직접 열리는 주소는 `wget`을 쓴다.


In [ ]:
ZIP_URL = "https://drive.google.com/file/d/1Ew_Yf3LktMzvtcFfgjsvshXbK-LUfqEl/view?usp=sharing"

TASK = "segmentation_medical"
ZIP_PATH = "/content/05_medical_segmentation.zip"
DST = f"/content/{TASK}"


## 2. 패키지


In [ ]:
!pip install -q gdown


In [ ]:
!apt-get -qq update
!apt-get -qq install -y fonts-noto-cjk fonts-nanum
from pathlib import Path
from PIL import ImageFont


def hangul_font(size=14):
    for path, index in (
        ("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc", 1),
        ("/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc", 1),
        ("/usr/share/fonts/truetype/nanum/NanumGothic.ttf", None),
        ("/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf", None),
    ):
        if not Path(path).exists():
            continue
        try:
            kw = {"index": index} if index is not None else {}
            font = ImageFont.truetype(path, size, **kw)
            print("hangul_font", path)
            return font
        except OSError:
            continue
    print("hangul_font missing. 범례는 영문으로 적는다.")
    return None


hangul_font()


## 3. zip 다운로드


In [ ]:
from pathlib import Path
import os


def fetch_zip(url, dest):
    url = (url or "").strip()
    if not url or "여기에" in url:
        raise SystemExit(
            "위 셀 ZIP_URL에 구글 드라이브 공유 링크(또는 직접 zip 주소)를 넣는다."
        )

    is_drive = "drive.google.com" in url or "docs.google.com" in url

    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 10_000:
        print("이미 받은 파일이 있다:", dest, dest.stat().st_size, "bytes")
        return

    dest.parent.mkdir(parents=True, exist_ok=True)

    os.environ["ZIP_URL"] = url
    os.environ["ZIP_PATH"] = str(dest)
    if is_drive:
        !gdown --fuzzy "$ZIP_URL" -O "$ZIP_PATH"
    else:
        !wget -c -O "$ZIP_PATH" "$ZIP_URL"

    print("받음", dest, dest.stat().st_size, "bytes")


fetch_zip(ZIP_URL, ZIP_PATH)


In [ ]:
!rm -rf {DST}


In [ ]:
!unzip -q -o {ZIP_PATH} -d /content


In [ ]:
from pathlib import Path
import torch
from IPython.display import display
from PIL import Image

ROOT = Path(DST)
if not ROOT.exists():
    raise SystemExit(f"{ROOT} 가 없다. zip 안에 과제 폴더가 있는지 확인한다.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT", ROOT)
print("torch", torch.__version__)
print("device", DEVICE)
if DEVICE.type == "cuda":
    print("gpu", torch.cuda.get_device_name(0))


def show_img(img, width=900):
    """셀 결과에 PIL 이미지를 띄운다. 너무 가로로 긴 장만 줄인다."""
    img = img.convert("RGB")
    w, h = img.size
    if w > width:
        img = img.resize((width, int(h * width / w)), Image.BILINEAR)
    display(img)


## 4. Label Visualization

복강경 담낭절제술(laparoscopic cholecystectomy)이다. 한 화면에 간(liver)·담낭(gallbladder)·지방(fat)·겸자(grasper)·L-hook가 같이 있다. 아래는 테스트 비디오에서 고른 강의용 세 장이다. 왼쪽이 입력이고 오른쪽이 정답을 색으로 칠한 그림이다. 테스트 세트 전체는 `dataset/test_img`이고 평가는 7번에서 한다.

색 범례를 먼저 보고 세 장의 입력과 정답을 나란히 본다. 그다음 숫자는 정답 마스크로 계산한 클래스 비율이다. 모델 점수가 아니라 어느 장기가 넓고 어느 장기가 좁은지를 보기 위한 숫자다.


In [ ]:
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

NUM_CLASSES = 13
CLASS_KO = [
    "배경",
    "복벽",
    "간",
    "위장관",
    "지방",
    "겸자",
    "결합조직",
    "혈액",
    "담낭관",
    "L-hook",
    "담낭",
    "간정맥",
    "간인대",
]
CLASS_EN = [
    "background",
    "abdominal wall",
    "liver",
    "GI tract",
    "fat",
    "grasper",
    "connective tissue",
    "blood",
    "cystic duct",
    "L-hook",
    "gallbladder",
    "hepatic vein",
    "liver ligament",
]
CLASS_LABEL = [
    ko if ko.lower() == en.lower() else f"{ko} ({en})"
    for ko, en in zip(CLASS_KO, CLASS_EN)
]
COLORS = np.array(
    [
        [127, 127, 127],
        [210, 140, 140],
        [255, 114, 114],
        [231, 70, 156],
        [186, 183, 75],
        [170, 255, 0],
        [255, 85, 0],
        [255, 0, 0],
        [255, 255, 0],
        [169, 255, 184],
        [255, 160, 165],
        [0, 50, 128],
        [111, 74, 0],
    ],
    dtype=np.uint8,
)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def overlay(img, mask, alpha=0.5):
    color = Image.fromarray(COLORS[np.clip(mask, 0, NUM_CLASSES - 1)], "RGB")
    if color.size != img.size:
        color = color.resize(img.size, Image.NEAREST)
    return Image.blend(img.convert("RGB"), color, alpha)


def hangul_font(size):
    for path, index in (
        ("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc", 1),
        ("/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc", 1),
        ("/usr/share/fonts/truetype/nanum/NanumGothic.ttf", None),
        ("/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf", None),
    ):
        if not Path(path).exists():
            continue
        try:
            kw = {"index": index} if index is not None else {}
            return ImageFont.truetype(path, size, **kw)
        except OSError:
            continue
    return None


def font(size):
    found = hangul_font(size)
    if found is not None:
        return found
    for path in (
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
    ):
        try:
            return ImageFont.truetype(path, size)
        except OSError:
            continue
    return ImageFont.load_default()


def caption(img, text):
    bar_h = 28
    canvas = Image.new("RGB", (img.width, img.height + bar_h), (20, 20, 20))
    canvas.paste(img.convert("RGB"), (0, bar_h))
    draw = ImageDraw.Draw(canvas)
    draw.text((8, 6), text, fill=(255, 255, 255), font=font(16))
    return canvas


def side_by_side(left, right):
    w, h = left.size
    canvas = Image.new("RGB", (w * 2, h))
    canvas.paste(left, (0, 0))
    canvas.paste(right, (w, 0))
    return canvas


print("클래스 색")
legend = ROOT / "examples" / "legend.png"
if legend.exists():
    show_img(Image.open(legend), width=360)
else:
    print("범례가 없다:", legend)

print()
print("강의용 세 장. 왼쪽 입력, 오른쪽 정답")
for path in sorted((ROOT / "examples").glob("[0-9][0-9].jpg")):
    img = Image.open(path).convert("RGB")
    mask_path = path.parent / f"{path.stem}_mask.png"
    mask = np.array(Image.open(mask_path)) if mask_path.exists() else None
    gt_path = path.parent / f"{path.stem}_gt.jpg"
    if gt_path.exists():
        gt_panel = Image.open(gt_path).convert("RGB")
        if gt_panel.size != img.size:
            gt_panel = gt_panel.resize(img.size, Image.BILINEAR)
    elif mask is not None:
        gt_panel = overlay(img, mask)
    else:
        gt_panel = img.copy()
    show_img(
        side_by_side(caption(img, f"{path.name}  input"), caption(gt_panel, "GT")),
        width=980,
    )
    if mask is None:
        print()
        continue
    h, w = mask.shape
    print(path.name, f"{w}x{h}")
    for c in range(1, NUM_CLASSES):
        px = int((mask == c).sum())
        if px == 0:
            continue
        print(f"  {CLASS_LABEL[c]}  {px:7d} px  ({px / (w * h):.1%})")
    print()


방금 본 장면은 object detection으로 풀기 어렵다. object detection은 객체마다 직사각형 박스 하나를 준다. 여기서는 그 박스만으로 부족하다.

1. **한 박스 안에 여러 장기가 들어간다.** 간(liver)을 감싸면 담낭(gallbladder)과 지방(fat)이 같이 들어간다.
2. **작은 구조는 박스가 거의 의미 없다.** 담낭관(cystic duct)·간정맥(hepatic vein)은 화면의 1%도 안 되는 경우가 많다.

그래서 픽셀마다 클래스를 칠하는 segmentation을 쓴다.


## 5. 신경망 모델

`torchvision`에는 DeepLabV3는 있고 DeepLabV3+는 없다. 아래는 ResNet-50 인코더에 ASPP와 디코더를 붙인 실습용 코드다. CholecSeg8K 원 논문도 이 구조를 베이스라인으로 썼다.

U-Net은 매 해상도에서 skip을 붙인다. DeepLabV3+는 가장 깊은 특징만 팽창률 1·6·12·18과 전역 평균으로 보고 가장자리용으로 layer1만 한 번 붙인다.

`show_deeplab_stack`이 그 경로를 그림으로 그린다.


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from torchvision.models import ResNet50_Weights, resnet50


class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, d=1):
        super().__init__()
        p = 0 if k == 1 else d
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, padding=p, dilation=d, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class ASPP(nn.Module):
    """같은 지도를 1x1, 팽창 6/12/18, 전역 평균으로 다섯 갈래에서 본다."""

    def __init__(self, in_ch=2048, out_ch=256, rates=(6, 12, 18)):
        super().__init__()
        self.branches = nn.ModuleList(
            [
                ConvBNReLU(in_ch, out_ch, k=1),
                *[ConvBNReLU(in_ch, out_ch, k=3, d=r) for r in rates],
            ]
        )
        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            ConvBNReLU(in_ch, out_ch, k=1),
        )
        self.project = nn.Sequential(
            ConvBNReLU(out_ch * 5, out_ch, k=1),
            nn.Dropout(0.1),
        )

    def forward(self, x):
        h, w = x.shape[-2:]
        parts = [b(x) for b in self.branches]
        pooled = F.interpolate(self.pool(x), size=(h, w), mode="bilinear", align_corners=False)
        return self.project(torch.cat(parts + [pooled], dim=1))


class DeepLabV3Plus(nn.Module):
    """output stride 16. layer4만 dilation, 저수준 skip은 layer1(1/4)이다."""

    def __init__(self, num_classes=13, pretrained=True):
        super().__init__()
        weights = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        backbone = resnet50(weights=weights, replace_stride_with_dilation=[False, False, True])
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.aspp = ASPP(2048, 256)
        self.low = ConvBNReLU(256, 48, k=1)
        self.decoder = nn.Sequential(
            ConvBNReLU(256 + 48, 256, k=3),
            ConvBNReLU(256, 256, k=3),
        )
        self.head = nn.Conv2d(256, num_classes, 1)

    def forward(self, x):
        h, w = x.shape[-2:]
        x = self.stem(x)
        low = self.layer1(x)
        x = self.layer2(low)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.aspp(x)
        x = F.interpolate(x, size=low.shape[-2:], mode="bilinear", align_corners=False)
        x = self.decoder(torch.cat([x, self.low(low)], dim=1))
        x = self.head(x)
        return F.interpolate(x, size=(h, w), mode="bilinear", align_corners=False)


def show_deeplab_stack(model, h=288, w=512):
    """더미 한 장을 통과시킨 뒤, ASPP와 layer1 skip을 그림으로 그린다."""
    from io import BytesIO

    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyBboxPatch
    from PIL import Image

    model.eval()
    x = torch.zeros(1, 3, h, w)
    with torch.no_grad():
        stem = model.stem(x)
        low = model.layer1(stem)
        l2 = model.layer2(low)
        l3 = model.layer3(l2)
        l4 = model.layer4(l3)
        aspp = model.aspp(l4)
        aspp_up = F.interpolate(aspp, size=low.shape[-2:], mode="bilinear", align_corners=False)
        dec = model.decoder(torch.cat([aspp_up, model.low(low)], dim=1))
        out = F.interpolate(model.head(dec), size=(h, w), mode="bilinear", align_corners=False)

    def label(t):
        _, c, hh, ww = t.shape
        return f"{c} ch\n{hh}×{ww}"

    fig, ax = plt.subplots(figsize=(12.4, 8.4))
    ax.set_xlim(0, 12)
    ax.set_ylim(0.2, 11.2)
    ax.axis("off")
    ax.set_title("DeepLabV3+   dashed = layer1 skip into decoder", fontsize=14, pad=8)
    boxes = {}

    def box(key, cx, cy, title, tensor, color):
        bw, bh = 2.05, 1.12
        patch = FancyBboxPatch(
            (cx - bw / 2, cy - bh / 2),
            bw,
            bh,
            boxstyle="round,pad=0.02,rounding_size=0.1",
            facecolor=color,
            edgecolor="#222",
            linewidth=1.2,
        )
        ax.add_patch(patch)
        ax.text(cx, cy + 0.22, title, ha="center", va="center", fontsize=8.5, fontweight="bold", color="#111")
        ax.text(cx, cy - 0.22, label(tensor), ha="center", va="center", fontsize=8, color="#222")
        boxes[key] = (cx, cy, bw, bh)

    def down(src, dst):
        x1, y1, _, h1 = boxes[src]
        x2, y2, _, h2 = boxes[dst]
        ax.annotate(
            "",
            xy=(x2, y2 + h2 / 2 + 0.02),
            xytext=(x1, y1 - h1 / 2 - 0.02),
            arrowprops=dict(arrowstyle="-|>", color="#333", lw=1.4, mutation_scale=10),
        )

    box("in", 2.0, 10.2, "input", x, "#ececec")
    box("stem", 2.0, 8.55, "stem  /4", stem, "#9ec0ea")
    box("low", 2.0, 6.9, "layer1  skip", low, "#7aa8de")
    box("l2", 2.0, 5.25, "layer2  /8", l2, "#568fd0")
    box("l3", 2.0, 3.6, "layer3  /16", l3, "#3a74b8")
    box("l4", 5.0, 1.7, "layer4  dilated", l4, "#4a4a6a")
    box("aspp", 8.2, 1.7, "ASPP  1,6,12,18", aspp, "#6b5b95")
    box("dec", 10.0, 6.9, "decoder", dec, "#2f9e66")
    box("head", 10.0, 10.2, "head  (classes)", out, "#ececec")
    down("in", "stem")
    down("stem", "low")
    down("low", "l2")
    down("l2", "l3")
    x1, y1, _, h1 = boxes["l3"]
    x2, y2, _, h2 = boxes["l4"]
    ax.annotate(
        "",
        xy=(x2 - 1.05, y2 + h2 / 2),
        xytext=(x1 + 0.15, y1 - h1 / 2),
        arrowprops=dict(arrowstyle="-|>", color="#333", lw=1.4, mutation_scale=10),
    )
    x3, y3, _, h3 = boxes["aspp"]
    ax.annotate(
        "",
        xy=(x3 - 1.05, y3),
        xytext=(x2 + 1.05, y2),
        arrowprops=dict(arrowstyle="-|>", color="#333", lw=1.4, mutation_scale=10),
    )
    ax.text(6.6, 2.15, "multi-scale", fontsize=7.5, color="#444", ha="center")
    x4, y4, w4, h4 = boxes["aspp"]
    x5, y5, w5, h5 = boxes["dec"]
    ax.annotate(
        "",
        xy=(x5, y5 - h5 / 2 - 0.02),
        xytext=(x4 + 0.2, y4 + h4 / 2),
        arrowprops=dict(arrowstyle="-|>", color="#333", lw=1.4, mutation_scale=10),
    )
    ax.text(9.35, 4.15, "upsample to 1/4", fontsize=7.5, color="#444", ha="center")
    x6, y6, w6, _ = boxes["low"]
    ax.annotate(
        "",
        xy=(x5 - w5 / 2 - 0.02, y5),
        xytext=(x6 + w6 / 2 + 0.02, y6),
        arrowprops=dict(arrowstyle="-|>", color="#d06000", lw=1.6, ls=(0, (5, 3)), mutation_scale=10),
    )
    ax.text((x6 + x5) / 2, y6 + 0.42, "skip concat", ha="center", fontsize=7.5, color="#d06000")
    x7, y7, _, h7 = boxes["dec"]
    x8, y8, _, h8 = boxes["head"]
    ax.annotate(
        "",
        xy=(x8, y8 - h8 / 2 - 0.02),
        xytext=(x7, y7 + h7 / 2 + 0.02),
        arrowprops=dict(arrowstyle="-|>", color="#333", lw=1.4, mutation_scale=10),
    )
    ax.text(x7 + 0.15, (y7 + y8) / 2, "×4", fontsize=7.5, color="#444", ha="left")
    n = sum(p.numel() for p in model.parameters())
    ax.text(6.0, 0.45, f"params {n:,}   ResNet-50 ImageNet encoder", ha="center", fontsize=9)
    fig.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=140, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    buf.seek(0)
    show_img(Image.open(buf), width=980)


show_deeplab_stack(DeepLabV3Plus(num_classes=NUM_CLASSES, pretrained=False).to("cpu"))


## 6. 학습

Dice는 예측한 칠과 정답 칠이 얼마나 겹치는지를 0에서 1 사이로 본 값이다. 교집합을 두 배로 하고 예측 픽셀 수와 정답 픽셀 수의 합으로 나눈다. 1이면 그 종류 픽셀이 같고 0이면 겹치는 픽셀이 없다.

픽셀 정확도로 고르지 않는 이유는 배경 때문이다. 화면의 대부분이 배경이면 전부 배경으로 찍어도 정확도는 높다. 교차엔트로피도 픽셀마다 틀렸는지를 보기 때문에 간(liver)·지방(fat)처럼 넓은 장기가 손실을 지배한다. Dice를 종류마다 구한 뒤 배경을 빼고 평균하면 담낭관(cystic duct)처럼 작은 장기도 간과 같은 비중으로 들어간다. 그래서 손실은 교차엔트로피와 그 Dice를 더한다.

에폭마다 검증 세트에서 배경을 뺀 평균 Dice를 보고 가장 높은 가중치를 `best_colab.pt`로 남긴다. 배경은 화면을 많이 차지해서 넣고 평균하면 점수가 항상 높아 보인다. 검증은 가중치를 고르는 자리일 뿐이고 최종 점수는 아니다. 테스트 비디오는 학습과 모델 선택에 쓰지 않는다.

이 셀은 손실과 체크포인트를 고르는 방법을 보여 주는 12 epoch다. T4에서 10분 안팎이다. zip의 `runs/best.pt`는 덮지 않는다. 7번 평가와 8번 시연은 그 파일을 읽는다. 시연만 하면 이 셀을 건너뛴다.


In [ ]:
import json
import random

import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import ColorJitter
from torchvision.transforms.functional import normalize, to_tensor

EPOCHS = 12
BATCH = 4
WIDTH, HEIGHT = 512, 288
LR = 3e-4
SEED = 42


class OrganDataset(Dataset):
    def __init__(self, img_dir, lab_dir, size, augment=False):
        self.images = sorted(img_dir.glob("*.jpg"))
        if not self.images:
            raise FileNotFoundError(img_dir)
        self.lab_dir = lab_dir
        self.size = size
        self.augment = augment
        self.jitter = ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.02)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_p = self.images[idx]
        lab_p = self.lab_dir / f"{img_p.stem}.png"
        img = Image.open(img_p).convert("RGB").resize(self.size, Image.BILINEAR)
        mask = Image.open(lab_p).resize(self.size, Image.NEAREST)
        if self.augment:
            if random.random() < 0.5:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
                mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
            img = self.jitter(img)
        x = normalize(to_tensor(img), IMAGENET_MEAN, IMAGENET_STD)
        y = torch.from_numpy(np.array(mask, dtype=np.int64)).clamp(0, NUM_CLASSES - 1)
        return x, y, img_p.name


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def class_weights(lab_dir):
    counts = np.zeros(NUM_CLASSES, dtype=np.float64)
    for path in lab_dir.glob("*.png"):
        mask = np.array(Image.open(path))
        counts += np.bincount(mask.ravel(), minlength=NUM_CLASSES)
    freq = counts / max(counts.sum(), 1.0)
    w = 1.0 / np.log(1.02 + freq)
    w[counts == 0] = 0.0
    pos = w[counts > 0]
    if pos.size:
        w = w / pos.mean()
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


def ce_dice(logits, target, weight=None):
    ce = F.cross_entropy(logits, target, weight=weight)
    pred = logits.softmax(dim=1)
    oh = F.one_hot(target, NUM_CLASSES).permute(0, 3, 1, 2).float()
    num = 2 * (pred * oh).sum(dim=(0, 2, 3))
    den = pred.sum(dim=(0, 2, 3)) + oh.sum(dim=(0, 2, 3))
    dice = 1 - (num + 1.0) / (den + 1.0)
    return ce + dice[1:].mean()


@torch.no_grad()
def eval_metrics(model, loader):
    model.eval()
    inter = torch.zeros(NUM_CLASSES)
    union = torch.zeros(NUM_CLASSES)
    pred_sum = torch.zeros(NUM_CLASSES)
    tgt_sum = torch.zeros(NUM_CLASSES)
    for x, y, _ in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        pred = model(x).argmax(dim=1)
        for c in range(NUM_CLASSES):
            p = pred == c
            t = y == c
            inter[c] += (p & t).sum().item()
            union[c] += (p | t).sum().item()
            pred_sum[c] += p.sum().item()
            tgt_sum[c] += t.sum().item()
    dice = (2 * inter + 1e-6) / (pred_sum + tgt_sum + 1e-6)
    iou = (inter + 1e-6) / (union + 1e-6)
    present = [c for c in range(1, NUM_CLASSES) if tgt_sum[c] > 0]
    return {
        "dice": float(dice[present].mean()) if present else 0.0,
        "iou": float(iou[present].mean()) if present else 0.0,
    }


set_seed(SEED)
size = (WIDTH, HEIGHT)
data = ROOT / "dataset"
runs = ROOT / "runs"
train_ds = OrganDataset(data / "train_img", data / "train_lab", size, augment=True)
val_ds = OrganDataset(data / "val_img", data / "val_lab", size, augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=0)

model = DeepLabV3Plus(num_classes=NUM_CLASSES, pretrained=True).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
weight = class_weights(data / "train_lab")
runs.mkdir(parents=True, exist_ok=True)
out_weights = runs / "best_colab.pt"
best_dice = -1.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    n = 0
    for x, y, _ in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = ce_dice(model(x), y, weight)
        loss.backward()
        optimizer.step()
        running += loss.item() * x.size(0)
        n += x.size(0)
    scheduler.step()
    metrics = eval_metrics(model, val_loader)
    print(
        f"epoch {epoch:03d}  loss {running / n:.4f}  "
        f"val dice {metrics['dice']:.3f} iou {metrics['iou']:.3f}"
    )
    if metrics["dice"] >= best_dice:
        best_dice = metrics["dice"]
        torch.save({"model": model.state_dict(), "size": size}, out_weights)

print("saved", out_weights, "best_val_dice", round(best_dice, 4))
print("zip best.pt unchanged", runs / "best.pt")


## 7. 테스트 평가

학습과 체크포인트에 쓰지 않은 비디오 전체다. 기본은 zip의 `best.pt`다. 6번에서 만든 가중치를 보려면 `WEIGHTS`를 `best_colab.pt`로 바꾼다.

첫 줄의 `dice`와 `iou`는 정답에 한 픽셀이라도 있는 장기·도구만 모아 평균한 값이다. 배경은 넣지 않는다. 둘 다 1에 가까울수록 칠이 정답과 잘 맞는다. IoU는 합집합 대비 교집합이라 Dice보다 낮게 나온다. 같은 방향으로 읽으면 된다.

그 아래 줄은 장기·도구마다 따로 적는다. Dice가 높으면 그 장기나 도구를 맞게 칠한 것이고 낮으면 빠지거나 다른 종류로 넘친 것이다. `gt_px`는 정답 픽셀 수다. 정답에는 없는데 Dice가 0이면 모델이 없는 종류를 칠한 경우다. 줄 자체가 없으면 정답에도 예측에도 그 종류가 없다.

평균은 종류마다 한 번씩 넣기 때문에 픽셀이 많은 간(liver)과 픽셀이 적은 L-hook이 같은 비중으로 들어간다. 그래서 간이 잘 나와도 작은 장기가 빠지면 평균이 내려간다.


In [ ]:
import json

import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms.functional import normalize, to_tensor


class OrganDataset(Dataset):
    def __init__(self, img_dir, lab_dir, size, augment=False):
        self.images = sorted(img_dir.glob("*.jpg"))
        if not self.images:
            raise FileNotFoundError(img_dir)
        self.lab_dir = lab_dir
        self.size = size
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_p = self.images[idx]
        lab_p = self.lab_dir / f"{img_p.stem}.png"
        img = Image.open(img_p).convert("RGB").resize(self.size, Image.BILINEAR)
        mask = Image.open(lab_p).resize(self.size, Image.NEAREST)
        x = normalize(to_tensor(img), IMAGENET_MEAN, IMAGENET_STD)
        y = torch.from_numpy(np.array(mask, dtype=np.int64)).clamp(0, NUM_CLASSES - 1)
        return x, y, img_p.name


@torch.no_grad()
def eval_metrics(model, loader):
    model.eval()
    inter = torch.zeros(NUM_CLASSES)
    union = torch.zeros(NUM_CLASSES)
    pred_sum = torch.zeros(NUM_CLASSES)
    tgt_sum = torch.zeros(NUM_CLASSES)
    for x, y, _ in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        pred = model(x).argmax(dim=1)
        for c in range(NUM_CLASSES):
            p = pred == c
            t = y == c
            inter[c] += (p & t).sum().item()
            union[c] += (p | t).sum().item()
            pred_sum[c] += p.sum().item()
            tgt_sum[c] += t.sum().item()
    dice = (2 * inter + 1e-6) / (pred_sum + tgt_sum + 1e-6)
    iou = (inter + 1e-6) / (union + 1e-6)
    present = [c for c in range(1, NUM_CLASSES) if tgt_sum[c] > 0]
    per_class = {}
    for c in range(NUM_CLASSES):
        per_class[CLASS_LABEL[c]] = {
            "dice": float(dice[c]),
            "iou": float(iou[c]),
            "gt_px": int(tgt_sum[c]),
            "pred_px": int(pred_sum[c]),
        }
    return {
        "n": len(loader.dataset),
        "dice": float(dice[present].mean()) if present else 0.0,
        "iou": float(iou[present].mean()) if present else 0.0,
        "per_class": per_class,
    }


WEIGHTS = ROOT / "runs" / "best.pt"
print("weights", WEIGHTS)
ckpt = torch.load(WEIGHTS, map_location="cpu", weights_only=True)
size = tuple(ckpt.get("size", (512, 288)))
model = DeepLabV3Plus(num_classes=NUM_CLASSES, pretrained=False)
model.load_state_dict(ckpt["model"])
model.to(DEVICE)
model.eval()

test_ds = OrganDataset(ROOT / "dataset" / "test_img", ROOT / "dataset" / "test_lab", size)
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=0)
metrics = eval_metrics(model, test_loader)
print(f"held_out_test  n={metrics['n']}  dice {metrics['dice']:.3f}  iou {metrics['iou']:.3f}")
report_pc = {}
for name, row in metrics["per_class"].items():
    if name == CLASS_LABEL[0]:
        continue
    if row["gt_px"] == 0 and row["pred_px"] == 0:
        continue
    print(
        f"  {name:24s}  dice {row['dice']:.3f}  iou {row['iou']:.3f}  "
        f"gt_px {row['gt_px']}"
    )
    report_pc[name] = {
        "dice": round(row["dice"], 4),
        "iou": round(row["iou"], 4),
        "gt_px": row["gt_px"],
        "pred_px": row["pred_px"],
    }
report = {
    "n": metrics["n"],
    "dice": round(metrics["dice"], 4),
    "iou": round(metrics["iou"], 4),
    "per_class": report_pc,
}
(ROOT / "runs").mkdir(parents=True, exist_ok=True)
(ROOT / "runs" / "test_metrics.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("saved", ROOT / "runs" / "test_metrics.json")


## 8. 추론 및 시각화

왼쪽부터 입력, 정답, DeepLabV3+ 예측, 범례다. 7번과 같이 zip의 `best.pt`를 읽는다. 6번에서 만든 가중치를 보려면 `WEIGHTS`를 `best_colab.pt`로 바꾼다. 간(liver)·복벽(abdominal wall)·담낭(gallbladder)처럼 넓은 장기는 이 세 장에서 잘 맞는다. 위장관(GI tract)은 간과 색이 비슷해서 7번 테스트 표에서 Dice가 가장 낮다. 여기 숫자는 이 장에 정답이 있는 클래스만 평균한 값이고, 테스트 세트 점수와 바꿔 읽지 않는다.


In [ ]:
import numpy as np
from pathlib import Path
import torch
from PIL import Image, ImageDraw, ImageFont
from torchvision.transforms.functional import normalize, to_tensor


def font_and_names(size):
    for path, index in (
        ("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc", 1),
        ("/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc", 1),
        ("/usr/share/fonts/truetype/nanum/NanumGothic.ttf", None),
        ("/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf", None),
    ):
        if not Path(path).exists():
            continue
        try:
            kw = {"index": index} if index is not None else {}
            return ImageFont.truetype(path, size, **kw), CLASS_LABEL
        except OSError:
            continue
    for path in (
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
    ):
        try:
            return ImageFont.truetype(path, size), CLASS_EN
        except OSError:
            continue
    return ImageFont.load_default(), CLASS_EN


def font(size):
    f, _ = font_and_names(size)
    return f


def caption(img, text):
    bar_h = 28
    canvas = Image.new("RGB", (img.width, img.height + bar_h), (20, 20, 20))
    canvas.paste(img, (0, bar_h))
    draw = ImageDraw.Draw(canvas)
    draw.text((8, 6), text, fill=(255, 255, 255), font=font(16))
    return canvas


def legend_panel(height):
    f, names = font_and_names(15)
    title, _ = font_and_names(16)
    col_w = 280
    img = Image.new("RGB", (col_w, height), (20, 20, 20))
    draw = ImageDraw.Draw(img)
    draw.text((8, 8), "classes", fill=(255, 255, 255), font=title)
    gap = max(18, (height - 40) // NUM_CLASSES)
    for i, name in enumerate(names):
        y = 36 + i * gap
        c = tuple(int(v) for v in COLORS[i])
        draw.rectangle([8, y, 24, y + 14], fill=c)
        draw.text((32, y - 2), name, fill=(230, 230, 230), font=f)
    return img


WEIGHTS = ROOT / "runs" / "best.pt"
print("weights", WEIGHTS)
ckpt = torch.load(WEIGHTS, map_location="cpu", weights_only=True)
size = tuple(ckpt.get("size", (512, 288)))
model = DeepLabV3Plus(num_classes=NUM_CLASSES, pretrained=False)
model.load_state_dict(ckpt["model"])
model.eval()
model.to(DEVICE)

out_dir = ROOT / "runs" / "infer"
out_dir.mkdir(parents=True, exist_ok=True)

for path in sorted((ROOT / "examples").glob("[0-9][0-9].jpg")):
    raw = Image.open(path).convert("RGB")
    img = raw.resize(size, Image.BILINEAR)
    x = normalize(to_tensor(img), IMAGENET_MEAN, IMAGENET_STD).to(DEVICE)
    with torch.no_grad():
        pred = model(x.unsqueeze(0)).argmax(dim=1)[0].cpu().numpy().astype(np.uint8)
    pred_ov = overlay(img, pred)
    mask_path = path.parent / f"{path.stem}_mask.png"
    gt = None
    if mask_path.exists():
        gt = np.array(Image.open(mask_path).resize(size, Image.NEAREST), dtype=np.uint8)
        gt_panel = overlay(img, gt)
    else:
        gt_panel = img.copy()
    panels = [
        caption(img, "input"),
        caption(gt_panel, "GT" if gt is not None else "GT missing"),
        caption(pred_ov, "DeepLabV3+"),
    ]
    w, h = panels[0].size
    leg = legend_panel(h)
    canvas = Image.new("RGB", (w * 3 + leg.width, h))
    for i, panel in enumerate(panels):
        canvas.paste(panel, (i * w, 0))
    canvas.paste(leg, (w * 3, 0))
    out = out_dir / f"{path.stem}_pred.jpg"
    canvas.save(out, quality=92)
    print(path.name)
    if gt is not None:
        rows = []
        for c in range(1, NUM_CLASSES):
            p = pred == c
            t = gt == c
            tgt = int(t.sum())
            if tgt == 0:
                continue
            den = int(p.sum()) + tgt
            d = (2 * int((p & t).sum()) / den) if den else 0.0
            rows.append((CLASS_LABEL[c], d, tgt))
        if rows:
            print(f"  mean_dice {sum(d for _, d, _ in rows) / len(rows):.3f}")
            for name, d, tgt in rows:
                print(f"  {name:24s}  dice {d:.3f}  gt_px {tgt}")
    show_img(canvas, width=980)
    print()
